# Notebook 10: Dataset Expansion & Activation Re-extraction

**Purpose:** Expand the IRIS dataset from 1,000 to ~5,000 balanced examples,
then re-extract GPT-2 activations and SAE features for the expanded dataset.

**Prerequisites:** Pre-trained SAE checkpoint (`sae_d6144_lambda1e-04.pt`) on Google Drive.

**Runtime:** Colab GPU (T4), ~15 minutes.

In [1]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -r /content/drive/MyDrive/iris/requirements.txt -q

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import sys
sys.path.insert(0, '/content/drive/MyDrive/iris')

from src.utils.helpers import set_seed, get_device
set_seed(42)
device = get_device()

Using GPU: Tesla T4


## Step 1: Expand the Dataset

Pull from HuggingFace datasets to grow from 1,000 to ~5,000 balanced examples.
This adds more injection categories and diverse normal prompts.

In [3]:
!pip install datasets -q

In [4]:
# Run the expansion script
%run /content/drive/MyDrive/iris/scripts/expand_dataset.py

IRIS Dataset Expansion
Loaded 1000 existing examples
Fetching deepset/prompt-injections...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/500 [00:00<?, ?B/s]

data/train-00000-of-00001-9564e8b05b4757(…):   0%|          | 0.00/40.3k [00:00<?, ?B/s]

data/test-00000-of-00001-701d16158af8736(…):   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

databricks-dolly-15k.jsonl:   0%|          | 0.00/13.1M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

  Got 2000 new normal examples
Fetching Open-Orca/OpenOrca (sampling 3000)...


README.md: 0.00B [00:00, ?B/s]

  Collected 500 normal examples...
  Collected 1000 normal examples...
  Collected 1500 normal examples...
  Collected 2000 normal examples...
  Collected 2500 normal examples...
  Collected 3000 normal examples...
  Got 3000 new normal examples

Total collected: 6000
  Labels: {0: 5500, 1: 500}
  Categories: {'instruction': 3500, 'mixed': 203, 'extraction': 74, 'roleplay': 74, 'override': 75, 'indirect': 74, 'closed_qa': 242, 'classification': 282, 'open_qa': 499, 'information_extraction': 188, 'brainstorming': 217, 'general_qa': 323, 'summarization': 159, 'creative_writing': 90}
  Sources: {'alpaca': 500, 'deepset_prompt_injections': 203, 'synthetic': 297, 'dolly': 2000, 'openorca': 3000}

Before balancing: 5500 normal, 500 injection
After balancing: 500 normal, 500 injection

Final Dataset Summary
Total: 1000
Labels: {1: 500, 0: 500}
Categories:
  instruction: 324 (normal=324, inject=0)
  mixed: 203 (normal=0, inject=203)
  override: 75 (normal=0, inject=75)
  extraction: 74 (normal

In [5]:
# Verify the expanded dataset
from src.data.dataset import IrisDataset
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/iris')
expanded_path = DRIVE_ROOT / 'data' / 'processed' / 'iris_dataset_expanded.json'

dataset = IrisDataset.load(expanded_path)
dataset.summary()

Loaded 1000 examples from /content/drive/MyDrive/iris/data/processed/iris_dataset_expanded.json
Total examples: 1000
  Labels:     {1: 500, 0: 500}
  Sources:    {'synthetic': 297, 'alpaca': 52, 'dolly': 176, 'deepset_prompt_injections': 203, 'openorca': 272}
  Categories: {'extraction': 74, 'override': 75, 'instruction': 324, 'open_qa': 42, 'mixed': 203, 'roleplay': 74, 'summarization': 15, 'creative_writing': 8, 'classification': 24, 'information_extraction': 17, 'general_qa': 31, 'indirect': 74, 'brainstorming': 17, 'closed_qa': 22}


## Step 2: Extract Activations for Expanded Dataset

Run GPT-2 on all prompts and extract residual stream activations at all 12 layers.
This gives us the raw representations needed for detection and SAE decomposition.

In [6]:
import torch
import numpy as np
from src.model.transformer import load_model, extract_activations

# Load GPT-2
model = load_model(device)

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Loaded pretrained model gpt2 into HookedTransformer
Loaded GPT-2 Small: 12 layers, d_model=768, vocab=50257


In [7]:
# Tokenize the expanded dataset
formatted_prompts = dataset.format_prompts()
print(f'Tokenizing {len(formatted_prompts)} prompts...')

tokens = model.tokenizer(
    formatted_prompts,
    padding=True,
    truncation=True,
    max_length=128,
    return_tensors='pt',
)
input_ids = tokens['input_ids']
attention_mask = tokens['attention_mask']
print(f'Token shape: {input_ids.shape}')

Tokenizing 1000 prompts...
Token shape: torch.Size([1000, 128])


In [8]:
# Extract activations at all 12 layers
# This takes ~5 minutes on a T4 for 5000 prompts
activations = extract_activations(
    model, input_ids, attention_mask,
    layers=list(range(12)),
    batch_size=32,
)

# Save to Drive
save_dict = {f'layer_{i}': activations[i] for i in range(12)}
save_dict['labels'] = np.array(dataset.labels)

save_path = DRIVE_ROOT / 'checkpoints' / 'expanded_activations.npz'
np.savez_compressed(str(save_path), **save_dict)
print(f'Saved activations to {save_path}')
print(f'File size: {save_path.stat().st_size / 1e6:.1f} MB')

Extracting activations: 100%|██████████| 32/32 [00:11<00:00,  2.81it/s]


Layer  0: shape=(1000, 768), mean=0.0000, std=1.7584
Layer  1: shape=(1000, 768), mean=0.0000, std=1.9708
Layer  2: shape=(1000, 768), mean=0.0000, std=2.1094
Layer  3: shape=(1000, 768), mean=0.0000, std=2.1717
Layer  4: shape=(1000, 768), mean=-0.0000, std=2.4154
Layer  5: shape=(1000, 768), mean=0.0000, std=2.6442
Layer  6: shape=(1000, 768), mean=0.0000, std=2.8722
Layer  7: shape=(1000, 768), mean=-0.0000, std=3.2677
Layer  8: shape=(1000, 768), mean=-0.0000, std=4.1522
Layer  9: shape=(1000, 768), mean=-0.0000, std=5.5192
Layer 10: shape=(1000, 768), mean=-0.0000, std=8.7741
Layer 11: shape=(1000, 768), mean=0.0000, std=20.2212
Saved activations to /content/drive/MyDrive/iris/checkpoints/expanded_activations.npz
File size: 34.2 MB


## Step 3: Compute SAE Features for Expanded Dataset

Run the trained SAE encoder on layer-0 activations to get the 6144-dim
sparse feature vectors for every prompt.

In [9]:
from src.sae.architecture import SparseAutoencoder
from src.utils.helpers import load_checkpoint
from src.analysis.features import compute_feature_activations, compute_sensitivity_scores

# Load the trained SAE (8x expansion, lambda=1e-4)
sae = SparseAutoencoder(d_input=768, expansion_factor=8, sparsity_coeff=1e-4)
ckpt_path = DRIVE_ROOT / 'checkpoints' / 'sae_d6144_lambda1e-04.pt'
load_checkpoint(ckpt_path, sae, device=device)
sae = sae.to(device)
sae.eval()

print(f'SAE loaded: {sae.d_input} -> {sae.d_sae}')

Checkpoint loaded: /content/drive/MyDrive/iris/checkpoints/sae_d6144_lambda1e-04.pt
  final_total_loss: 0.6553142070770264
  final_mse_loss: 0.6537281423807144
  final_l1_loss: 1.586078554391861
  final_mean_sparsity: 0.2865375429391861
  input_variance: 4.018444061279297
  j2_threshold: 0.4018444061279297
SAE loaded: 768 -> 6144


In [10]:
# Compute SAE features on layer-0 activations
layer0_acts = activations[0]  # shape: (N, 768)
feature_matrix = compute_feature_activations(sae, layer0_acts, device=device)
print(f'Feature matrix shape: {feature_matrix.shape}')

# Compute sensitivity scores
labels = np.array(dataset.labels)
sensitivity = compute_sensitivity_scores(feature_matrix, labels)

# Save
np.save(str(DRIVE_ROOT / 'checkpoints' / 'expanded_feature_matrix.npy'), feature_matrix)
np.save(str(DRIVE_ROOT / 'checkpoints' / 'expanded_sensitivity_scores.npy'), sensitivity)
print('Saved feature matrix and sensitivity scores')

Feature matrix shape: (1000, 6144)
Sensitivity scores computed for 6144 features:
  Injection-associated (positive): 3052
  Normal-associated (negative):    2740
  Neutral (zero):                  352
  Max sensitivity:  0.4294
  Min sensitivity:  -0.3909
  Mean |sensitivity|: 0.0494
Saved feature matrix and sensitivity scores


## Step 4: Quick Validation

Run a quick detection comparison to verify the expanded dataset works.

In [11]:
from sklearn.model_selection import train_test_split
from src.analysis.detection import run_detection_comparison

# 70/30 split
n = len(labels)
indices = np.arange(n)
train_idx, test_idx = train_test_split(
    indices, train_size=0.7, stratify=labels, random_state=42
)

texts = dataset.texts
results, preds = run_detection_comparison(
    train_texts=[texts[i] for i in train_idx],
    train_labels=labels[train_idx],
    test_texts=[texts[i] for i in test_idx],
    test_labels=labels[test_idx],
    train_activations=layer0_acts[train_idx],
    test_activations=layer0_acts[test_idx],
    train_features=feature_matrix[train_idx],
    test_features=feature_matrix[test_idx],
)

Training Approach 1: TF-IDF + Logistic Regression...
  -> F1 = 0.9550, AUC = 0.9838
Training Approach 1b: TF-IDF + Random Forest...
  -> F1 = 0.9477, AUC = 0.9824
Training Approach 2: Raw Activation + Logistic Regression...
  -> F1 = 0.8837, AUC = 0.9267
Training Approach 3: SAE Features (all) + Logistic Regression...
  -> F1 = 0.8779, AUC = 0.9301

-----------------------------------------------------------------------------------------
Approach                     Precision   Recall      F1          Accuracy    Roc_auc     
-----------------------------------------------------------------------------------------
TF-IDF + LogReg              0.9928 *    0.9200 *    0.9550 *    0.9567 *    0.9838 *    
TF-IDF + RandomForest        0.9927      0.9067      0.9477      0.9500      0.9824      
Raw Activation + LogReg      0.8808      0.8867      0.8837      0.8833      0.9267      
SAE Features (all) + LogReg  0.8693      0.8867      0.8779      0.8767      0.9301      
------------------

In [12]:
# Save quick-check results
import json

results_path = DRIVE_ROOT / 'results' / 'metrics' / 'c3_expanded_quick_check.json'
results_path.parent.mkdir(parents=True, exist_ok=True)
results_path.write_text(json.dumps(results, indent=2))
print(f'Saved to {results_path}')

print('\n--- Dataset expansion complete! ---')
print(f'Dataset: {len(dataset)} examples')
print(f'Activations: {layer0_acts.shape}')
print(f'SAE features: {feature_matrix.shape}')
print(f'Sensitivity scores: {sensitivity.shape}')

Saved to /content/drive/MyDrive/iris/results/metrics/c3_expanded_quick_check.json

--- Dataset expansion complete! ---
Dataset: 1000 examples
Activations: (1000, 768)
SAE features: (1000, 6144)
Sensitivity scores: (6144,)
